In [1]:
!nvidia-smi

Wed Nov 25 09:42:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Example: Move a model to the GPU
# model.to(device)

Using device: cuda


In [3]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/essay_scoring
!pip install -q streamlit peft bitsandbytes accelerate
!ls models

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/essay_scoring
fast_heads.joblib  llama3_aes_lora


In [4]:
path = "/content/drive/MyDrive/essay_scoring/scorer.py"
code = open(path).read()
code = code.replace(
    'model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")',
    'model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map={"": 0}, torch_dtype=torch.float16)')
open(path, "w").write(code)
print("device_map={\"\": 0}" in code)

True


In [7]:
!nohup streamlit run app.py --server.port 8501 > log.txt 2>&1 &
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501

2026-11-25T12:02:35Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-25T12:02:35Z INF Requesting new quick Tunnel on trycloudflare.com...
^C


In [16]:
!du -sh ~/.cache/huggingface
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!tail -n 5 /content/drive/MyDrive/essay_scoring/log.txt

5.4G	/root/.cache/huggingface
memory.used [MiB], memory.total [MiB]
5723 MiB, 15360 MiB
If you are trying to access a private or gated repo, make sure you are authenticated and
your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
  Stopping...


In [22]:
!find /content/drive/MyDrive -name "adapter_model.safetensors" -exec ls -l {} \;

-rw------- 1 root root 20971520 Nov 25 08:54 /content/drive/MyDrive/essay_scoring/models/llama3_aes_lora/adapter_model.safetensors


In [28]:
!pkill -f streamlit
!cp -r /content/drive/MyDrive/essay_scoring /content/essay_scoring
%cd /content/essay_scoring
!ls -lh models/llama3_aes_lora/

/content/essay_scoring
total 44M
-rw------- 1 root root 1.4K Nov 25 09:12 adapter_config.json
-rw------- 1 root root  27M Nov 25 09:12 adapter_model.safetensors
-rw------- 1 root root 5.4K Nov 25 09:12 README.md
-rw------- 1 root root  52K Nov 25 09:12 tokenizer_config.json
-rw------- 1 root root  17M Nov 25 09:12 tokenizer.json


In [37]:
!ls -lh /content/essay_scoring/models/llama3_aes_lora/adapter_model.safetensors

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
-rw-r--r-- 1 root root 161M Nov 25 09:23 /content/essay_scoring/models/llama3_aes_lora/adapter_model.safetensors


In [35]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [5]:
!pkill -f streamlit
!cp -f /content/drive/MyDrive/essay_scoring/models/llama3_aes_lora/adapter_model.safetensors /content/essay_scoring/models/llama3_aes_lora/adapter_model.safetensors
!ls -lh /content/essay_scoring/models/llama3_aes_lora/adapter_model.safetensors

%cd /content/essay_scoring
!nohup streamlit run app.py --server.port 8501 > log.txt 2>&1 &

-rw-r--r-- 1 root root 161M Nov 25 09:42 /content/essay_scoring/models/llama3_aes_lora/adapter_model.safetensors
/content/essay_scoring
